In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
import os

In [2]:
df = pd.read_csv('data/creditcard.csv')

In [3]:
predictors = ['Time','V1','V2','V3','V4','V5','V6','V7','V8','V9','V10',
              'V11','V12','V13','V14','V15','V16','V17','V18','V19',
              'V20','V21','V22','V23','V24','V25','V26','V27','V28','Amount']
target = 'Class'

### Split data — 70% old (reference), 30% new (current)
#### Split by time because Time column = actual transaction order
#### First 70% = what model will train on (reference period)
#### Last 30%  = what will be used to simulate drift later

In [4]:
split_point = int(len(df) * 0.7)
reference_data = df.iloc[:split_point].copy()
current_data   = df.iloc[split_point:].copy()

### XGBoost Model

In [5]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(reference_data, test_size=0.2, random_state=2018)


In [6]:
dtrain = xgb.DMatrix(train_df[predictors], train_df[target].values)
dtest  = xgb.DMatrix(test_df[predictors],  test_df[target].values)

In [7]:
params = {
    'objective':        'binary:logistic',
    'eta':              0.039,
    'max_depth':        2,
    'subsample':        0.8,
    'colsample_bytree': 0.9,
    'eval_metric':      'auc',
    'random_state':     2018
}

In [8]:
model = xgb.train(params, dtrain, num_boost_round=200,
                  evals=[(dtest, 'test')], verbose_eval=50)

[0]	test-auc:0.91962
[50]	test-auc:0.96476
[100]	test-auc:0.96990
[150]	test-auc:0.97508
[199]	test-auc:0.97801


### Save model and reference data

In [10]:
os.makedirs('models', exist_ok=True)

In [11]:
with open('models/xgb_model.pkl', 'wb') as f:
    pickle.dump(model, f)

In [12]:
reference_data[predictors].to_pickle('models/reference_data.pkl')

In [13]:
print(f"   Reference data shape: {reference_data.shape}")

   Reference data shape: (199364, 31)
